# Surface Energy

Calculate the surface energy of a slab using a DFT workflow on the Mat3ra platform.

This notebook resolves the bulk material from slab metadata and checks that both the bulk material and its refined `total_energy` property already exist on the platform.

If either the bulk material or bulk total energy is missing, the notebook stops and points to the Total Energy notebook. It does not create or run a bulk pre-calculation automatically.

<h2 style="color:green">Usage</h2>

1. Put a slab material JSON into `../uploads`, or set `SLAB_NAME` to match an existing slab on the platform.
2. Set the slab and workflow parameters in cells 1.2 and 1.3 below.
3. Click "Run" > "Run All".
4. If the bulk material or bulk total energy is missing, run the Total Energy notebook for the exact bulk material first, then rerun this notebook.
5. Inspect the resolved bulk, the bulk total energy used by the workflow, and the final surface energy.

## Summary

1. Set up the environment and parameters.
2. Authenticate and initialize API client.
3. Load slab, resolve the exact bulk candidate, and check that exact bulk exists on the platform.
4. Check that refined bulk total energy already exists.
5. Configure and save the Surface Energy workflow.
6. Configure compute.
7. Create, submit, and monitor the Surface Energy job.
8. Retrieve results.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
SLAB_NAME = "Slab"  # Name of the slab to load from uploads folder or from the platform

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "surface_energy.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Surface Energy"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds


### 1.3. Set specific surface energy parameters

In [ ]:
# K-grid for slab SCF (if not set, KPPRA is used by default)
SCF_KGRID = None  # e.g., [4, 4, 1]

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable `OIDC_ACCESS_TOKEN`.


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account to work under


In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project


In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load slab and resolve the exact bulk candidate
### 3.1. Load slab from local file or platform


In [ ]:
import re
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

slab = load_material_from_folder(FOLDER, SLAB_NAME)

if slab is None:
    slab_matches = client.materials.list({
        "name": {"$regex": re.escape(SLAB_NAME), "$options": "i"},
    })
    if not slab_matches:
        raise ValueError(
            f"No slab containing '{SLAB_NAME}' was found in '{FOLDER}' or on the platform."
        )
    slab = Material.create(slab_matches[0])
    print(f"♻️  Loaded slab from platform: {slab_matches[0]['_id']}")
else:
    print(f"✅ Loaded slab from folder: {slab.name}")

visualize(slab)

### 3.2. Load workflow from Standata


In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow

surface_workflow_config = WorkflowStandata.filter_by_application(APPLICATION_NAME).get_by_name_first_match(
    WORKFLOW_SEARCH_TERM
)
surface_workflow = Workflow.create(surface_workflow_config)
resolve_bulk_by_id_subworkflow = surface_workflow.subworkflows[1]
resolve_bulk_by_build_subworkflow = surface_workflow.subworkflows[2]
get_bulk_energy_subworkflow = surface_workflow.subworkflows[3]
print(f"Loaded workflow: {surface_workflow.name}")

### 3.3. Check for the exact bulk material from slab metadata

In [ ]:
# Extract bulk lookup queries from the workflow definition so they stay in sync
# with what the workflow actually uses at runtime.
from mat3ra.notebooks_utils.ui import display_JSON

SLAB = slab.to_dict()
bulk_id_template = resolve_bulk_by_id_subworkflow.get_unit_by_name(name="assign-bulk-id").value
BULK_ID = eval(bulk_id_template, {"__builtins__": {}}, {"SLAB": SLAB})

if BULK_ID is not None:
    io_unit = resolve_bulk_by_id_subworkflow.get_unit_by_name(name="io-bulk")
    query_template = io_unit.input[0]["endpoint_options"]["params"]["query"]
    bulk_query = eval(query_template, {"__builtins__": {}}, {"BULK_ID": BULK_ID})
else:
    assign_crystal_template = resolve_bulk_by_build_subworkflow.get_unit_by_name(name="assign-crystal").value
    LAST_BUILD_CRYSTAL = eval(assign_crystal_template, {"__builtins__": {}}, {"SLAB": SLAB})
    bulk_query_template = resolve_bulk_by_build_subworkflow.get_unit_by_name(name="assign-bulk-query").value
    bulk_query = eval(bulk_query_template, {"__builtins__": {}}, {"LAST_BUILD_CRYSTAL": LAST_BUILD_CRYSTAL})

if bulk_query is None:
    raise ValueError(
        "Surface slab build metadata does not carry a crystal scaledHash or hash. "
        "Upload the bulk material and run total_energy.ipynb first."
    )

projection_template = resolve_bulk_by_build_subworkflow.get_unit_by_name(name="io-bulk").input[0][
    "endpoint_options"
]["params"]["projection"]
projection = eval(projection_template, {"__builtins__": {}})
bulk_matches = client.materials.list(query=bulk_query, projection=projection)
if not bulk_matches:
    raise ValueError(
        "The bulk material resolved from slab metadata is not on the platform. "
        "Run total_energy.ipynb for that bulk material first, then rerun this notebook."
    )

bulk_material_response = bulk_matches[0]
bulk_material = Material.create(bulk_material_response)
print(f"Found exact bulk material: {bulk_material_response['_id']}")
print(f"Resolved bulk query: {bulk_query}")
display_JSON({"bulk_query": bulk_query}, level=2)

### 3.4. Inspect the exact bulk material found on the platform


In [ ]:
display_JSON(bulk_material_response, level=2)
visualize(bulk_material)


### 3.5. Save the slab material for the workflow


In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_slab_response = get_or_create_material(client, slab, ACCOUNT_ID)

print(f"✅ Saved slab material: {saved_slab_response['_id']}")
saved_slab = Material.create(saved_slab_response)

## 4. Check bulk total energy
### 4.1. Check refined bulk total energy for the exact bulk material


In [ ]:
import json

# Extract the bulk TE query and projection from the workflow definition
# so they stay in sync with what the workflow actually uses at runtime.
io_unit = get_bulk_energy_subworkflow.get_unit_by_name(name="io-e-bulk")
query_template = io_unit.input[0]["endpoint_options"]["params"]["query"]
projection_template = io_unit.input[0]["endpoint_options"]["params"]["projection"]

BULK = bulk_material_response
query = eval(query_template, {"__builtins__": {}}, {"BULK": BULK})
projection = eval(projection_template, {"__builtins__": {}})
results = client.properties.request(
    "GET",
    "refined-properties",
    params={
        "query": json.dumps(query),
        "projection": json.dumps(projection),
    },
)
bulk_total_energy_property = results[0] if results else None
if bulk_total_energy_property is None:
    raise RuntimeError(
        "Bulk total energy does not exist for the resolved exact bulk material. "
        "Run total_energy.ipynb first."
    )

print(
    f"♻️  Found refined bulk total energy: {bulk_total_energy_property['_id']} -> "
    f"{bulk_total_energy_property['data']['value']}"
)
display_JSON(bulk_total_energy_property, level=2)


## 5. Configure the Surface Energy workflow
### 5.1. Select application


In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")


### 5.2. Configure workflow and preview it


In [ ]:
from mat3ra.wode.context.providers import PointsGridDataProvider
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow


def apply_workflow_kgrids(workflow: Workflow, scf_kgrid=None) -> Workflow:
    if scf_kgrid is not None:
        new_context_scf = PointsGridDataProvider(dimensions=scf_kgrid, isEdited=True).get_context_item_data()
        for subworkflow in workflow.subworkflows:
            unit_names = [unit.name for unit in subworkflow.units]
            if "pw_scf" not in unit_names:
                continue
            unit_to_modify_scf = subworkflow.get_unit_by_name(name="pw_scf")
            unit_to_modify_scf.add_context(new_context_scf)
            subworkflow.set_unit(unit_to_modify_scf)
            break
    return workflow


surface_workflow.name = MY_WORKFLOW_NAME
surface_workflow = apply_workflow_kgrids(surface_workflow, scf_kgrid=SCF_KGRID)

visualize_workflow(surface_workflow)

### 5.3. Save workflow to collection


In [ ]:
saved_surface_workflow_response = get_or_create_workflow(client, surface_workflow, ACCOUNT_ID)
saved_surface_workflow = Workflow.create(saved_surface_workflow_response)
print(f"Surface workflow ID: {saved_surface_workflow.id}")


## 6. Create the compute configuration
### 6.1. Select cluster


In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 6.2. Create compute configuration


In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


## 7. Create the Surface Energy job
### 7.1. Create job with slab and workflow configuration


In [ ]:
from mat3ra.notebooks_utils.job import create_job, wait_for_jobs_to_finish_async
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.ui import display_JSON

print(f"Slab material: {saved_slab.id}")
print(f"Resolved exact bulk material: {bulk_material.id}")
print(f"Surface workflow: {saved_surface_workflow.id}")
print(f"Project: {project_id}")

surface_job_name = f"{MY_WORKFLOW_NAME} {saved_slab.formula} {timestamp}"
surface_job_response = create_job(
    api_client=client,
    materials=[saved_slab],
    workflow=surface_workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=surface_job_name,
    compute=compute.to_dict(),
)

surface_job = dict_to_namespace_recursive(surface_job_response)
surface_job_id = surface_job._id
print(f"✅ Surface Energy job created successfully: {surface_job_id}")
display_JSON(surface_job_response)

### 7.2. Submit the Surface Energy job and monitor the status


In [ ]:
client.jobs.submit(surface_job_id)
print(f"✅ Job {surface_job_id} submitted successfully!")


In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, [surface_job_id], poll_interval=POLL_INTERVAL)


## 8. Retrieve results
### 8.1. Retrieve and visualize Surface Energy results


In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

surface_energy_data = client.properties.get_for_job(surface_job_id, property_name="surface_energy")
slab_total_energy_data = client.properties.get_for_job(
    surface_job_id, property_name=PropertyName.scalar.total_energy.value
)

if surface_energy_data:
    visualize_properties(surface_energy_data, title="Surface Energy")
else:
    print("No 'surface_energy' property was returned for the job.")

if slab_total_energy_data:
    visualize_properties(slab_total_energy_data, title="Slab Total Energy")
else:
    print("No slab total energy property was returned for the job.")

print(f"Resolved bulk query from slab metadata: {bulk_query}")
print(f"Exact bulk material used by the workflow: {bulk_material_response['_id']}")
print(f"Exact bulk exabyteId: {bulk_material_response['exabyteId']}")
print(f"Refined bulk total energy used by the workflow: {bulk_total_energy_property['data']['value']}")
print(f"Saved slab material used by the workflow: {saved_slab_response['_id']}")